# Lekcija 12: Deriviranje i detekcija rubova

Rub je, neformalno, mjesto na kojem se intenzitet brzo mijenja. To je tvrdnja o *derivaciji*. U ovoj ćemo lekciji izgraditi diskretne derivacije slike &mdash; Prewittov, Sobelov i Scharrov operator &mdash; kao konvolucijske jezgre iz Lekcije 10, koristiti ih u klasičnom **Cannyjevu** detektoru rubova te na kraju dobivene karte rubova pretvoriti u konkretne oblike pomoću **Houghove transformacije**.


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Od derivacija do gradijenata

Promatrajmo sliku kao funkciju $I(x,y)$. Njezin gradijent $\nabla I = (I_x, I_y)$ pokazuje smjer najbržeg porasta intenziteta, pri čemu su $I_x = \partial I/\partial x$ i $I_y = \partial I/\partial y$ **parcijalne derivacije** intenziteta slike u vodoravnom i okomitom smjeru &mdash; opisuju brzinu promjene intenziteta pri pomaku za jedan piksel udesno ili prema dolje. Dva broja sažimaju gradijent u svakom pikselu:

- **iznos** $\|\nabla I\| = \sqrt{I_x^2 + I_y^2}$ &mdash; koliko je promjena nagla; velik je na rubovima, a blizu nule u jednolikim područjima
- **smjer** $\theta = \mathrm{atan2}(I_y, I_x)$ &mdash; smjer najbržeg porasta intenziteta, okomit na rub

Vrijednosti $I_x$ i $I_y$ izračunavaju se konvolucijom slike s malim jezgrama za deriviranje.


In [ ]:
def make_test_image(size=200):
    img = np.zeros((size, size), dtype=np.uint8)
    cv2.rectangle(img, (30, 30), (100, 100), 200, -1)
    cv2.circle(img, (140, 140), 40, 150, -1)
    pts = np.array([[150, 30], [190, 90], [110, 90]], dtype=np.int32)
    cv2.fillPoly(img, [pts], 25)  # low-contrast triangle, for the threshold demo later
    return img

img = make_test_image()
plt.imshow(img, cmap='gray')
plt.title('Test image (edges at several orientations)')
plt.axis('off')
plt.show()

## Jednostavna derivacija: samo razlika

Najjednostavnija procjena $I_x$ koristi jednodimenzionalnu jezgru središnje razlike $[-1,\ 0,\ 1]$. Ona radi, ali bez zaglađivanja u okomitom smjeru vrlo je osjetljiva na šum.


In [ ]:
simple_x = np.array([[-1, 0, 1]], dtype=np.float64)
simple_y = simple_x.T

gx = cv2.filter2D(img.astype(np.float64), cv2.CV_64F, simple_x)
gy = cv2.filter2D(img.astype(np.float64), cv2.CV_64F, simple_y)
magnitude = np.sqrt(gx**2 + gy**2)

plt.imshow(magnitude, cmap='gray')
plt.title('Gradient magnitude, simple [-1, 0, 1] kernel')
plt.axis('off')
plt.show()

## Prewitt, Sobel i Scharr: derivacija i zaglađivanje

Praktični operatori kombiniraju razliku u jednom smjeru s prosjekom za *zaglađivanje* u okomitom smjeru &mdash; upravo to ih čini otpornima na šum. Razlikuju se samo po težinama tog zaglađivanja:

| Operator | Jezgra za smjer $x$ | Težine u okomitom smjeru |
|---|---|---|
| **Prewitt** | $\begin{bmatrix}-1&0&1\\-1&0&1\\-1&0&1\end{bmatrix}$ | jednake: 1, 1, 1 |
| **Sobel** | $\begin{bmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{bmatrix}$ | binomne: 1, 2, 1 |
| **Scharr** | $\begin{bmatrix}-3&0&3\\-10&0&10\\-3&0&3\end{bmatrix}$ | 3, 10, 3 |

Svaka jezgra za smjer $y$ samo je transponirana jezgra za smjer $x$. Sobelove binomne težine aproksimiraju malu Gaussovu jezgru, zbog čega je on najčešći početni izbor. Scharrove težine odabrane su numeričkom optimizacijom kako bi *procijenjeni smjer gradijenta* bio što točniji pri različitim rotacijama &mdash; odnosno što bliži stvarnoj kontinuiranoj derivaciji koliko to dopušta cjelobrojna jezgra veličine $3\times3$. To je posebno važno u primjenama poput optičkog toka, koje ovise o preciznim kutovima gradijenta, a ne samo o položaju ruba.


In [ ]:
prewitt_x = np.array([[-1, 0, 1]] * 3, dtype=np.float64)
prewitt_y = prewitt_x.T

operators = {
    'Simple': (simple_x, simple_y),
    'Prewitt': (prewitt_x, prewitt_y),
    'Sobel': None,   # use cv2.Sobel directly below
    'Scharr': None,  # use cv2.Scharr directly below
}

img_f = img.astype(np.float64)
magnitudes = {}
magnitudes['Simple'] = magnitude
magnitudes['Prewitt'] = np.sqrt(cv2.filter2D(img_f, cv2.CV_64F, prewitt_x)**2 +
                                 cv2.filter2D(img_f, cv2.CV_64F, prewitt_y)**2)
magnitudes['Sobel'] = np.sqrt(cv2.Sobel(img_f, cv2.CV_64F, 1, 0, ksize=3)**2 +
                               cv2.Sobel(img_f, cv2.CV_64F, 0, 1, ksize=3)**2)
magnitudes['Scharr'] = np.sqrt(cv2.Scharr(img_f, cv2.CV_64F, 1, 0)**2 +
                                cv2.Scharr(img_f, cv2.CV_64F, 0, 1)**2)

fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
for ax, (name, mag) in zip(axes, magnitudes.items()):
    ax.imshow(mag, cmap='gray')
    ax.set_title(name, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

Na čistoj slici s malo šuma poput ove sva četiri rezultata izgledaju gotovo identično &mdash; praktične su razlike suptilne i uglavnom dolaze do izražaja uz šum ili kada je potreban precizan kut na razini manjoj od piksela, a ne kao vidljivo drukčija karta rubova.


### Otpornost na šum: zašto je važno zaglađivanje u okomitom smjeru

Kako bismo konkretno pokazali korist zaglađivanja u okomitom smjeru, mjerimo odziv svake jezgre na čisti šum u jednolikom području bez rubova. Za poštenu usporedbu prvo skaliramo svaku jezgru tako da ima *isto pojačanje* na idealnom skokovitom rubu, odnosno dijelimo je zbrojem pozitivnih težina &mdash; inače bi jezgra s većim izvornim koeficijentima izgledala šumnije zbog ukupnog faktora skaliranja, a ne zbog svojeg oblika.


In [ ]:
rng = np.random.default_rng(0)
flat_noisy = np.full((150, 150), 128.0) + rng.normal(0, 15, (150, 150))

kernels = {
    'Simple [-1,0,1]': simple_x,
    'Prewitt': prewitt_x,
    'Sobel': np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float64),
    'Scharr': np.array([[-3, 0, 3], [-10, 0, 10], [-3, 0, 3]], dtype=np.float64),
}

print(f'{"kernel":>18} {"noise std (gain-normalized)":>30}')
for name, k in kernels.items():
    gain = k[k > 0].sum()
    response = cv2.filter2D(flat_noisy, cv2.CV_64F, k / gain)
    print(f'{name:>18} {response.std():>30.2f}')

Jezgra obične razlike daje primjetno više šuma od bilo kojeg operatora veličine $3\times3$ &mdash; usrednjavanje preko 3 retka ili stupca tijekom računanja razlike znatno smanjuje odziv na šum, što je upravo svrha uključivanja zaglađivanja u jezgru za deriviranje.


## Od gradijenata do rubova: Cannyjev detektor

Jednostavno pragovanje iznosa gradijenta daje debela, šumna područja rubova. **Cannyjev** detektor rubova (<a href="../references.html#canny-1986">Canny, 1986</a><span class="landmark-paper">&#9733;</span>) poboljšava rezultat kroz više koraka:

1. **Zaglađivanje** Gaussovim filtrom radi suzbijanja šuma prije deriviranja.
2. **Izračun gradijenata**, interno Sobelovim operatorom, za dobivanje iznosa i smjera u svakom pikselu.
3. **Potiskivanje nemaksimalnih vrijednosti**: u svakom pikselu zadržava se iznos gradijenta samo ako je lokalni maksimum *duž smjera gradijenta* &mdash; time se široki grebeni gradijenta stanjuju na linije široke jedan piksel.
4. **Dvostruko pragovanje i histereza**: pikseli iznad visokog praga sigurni su rubovi; pikseli ispod niskog praga odbacuju se; pikseli između tih pragova zadržavaju se samo ako su povezani sa sigurnim rubom. Time se povezuju slabi, ali stvarni dijelovi rubova i potiskuju izolirani odzivi na šum.


In [ ]:
calvin = cv2.imread('../img/calvin.png', cv2.IMREAD_GRAYSCALE)

calvin_sobel_mag = np.sqrt(cv2.Sobel(calvin.astype(np.float64), cv2.CV_64F, 1, 0, ksize=3)**2 +
                           cv2.Sobel(calvin.astype(np.float64), cv2.CV_64F, 0, 1, ksize=3)**2)
naive_edges = (calvin_sobel_mag > 150).astype(np.uint8) * 255  # crude: just threshold |gradient|
canny_edges = cv2.Canny(calvin, threshold1=80, threshold2=160)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, im, title in zip(axes, [calvin, naive_edges, canny_edges],
                          ['Original', 'Naive: threshold |gradient|\n(thick, blobby)', 'Canny\n(thin, connected)']):
    ax.imshow(im, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

<p class="image-attribution">Izvor slike: <a href="https://www.nga.gov/artworks/37890-john-calvin" target="_blank" rel="noopener">National Gallery of Art</a></p>


### Osjetljivost na pragove

Dva Cannyjeva praga određuju kompromis između potpunosti rubova i šuma. Ako su preniski, šum se prepoznaje kao lažni rubovi; ako su previsoki, stvarni rubovi slabog kontrasta ostaju neotkriveni.


In [ ]:
noisy_calvin = np.clip(calvin.astype(np.float64) + rng.normal(0, 8, calvin.shape), 0, 255).astype(np.uint8)

threshold_pairs = [(20, 60), (80, 160), (150, 220)]

fig, axes = plt.subplots(1, len(threshold_pairs), figsize=(10, 3.5))
for ax, (lo, hi) in zip(axes, threshold_pairs):
    edges = cv2.Canny(noisy_calvin, lo, hi)
    ax.imshow(edges, cmap='gray')
    ax.set_title(f'thresholds=({lo}, {hi})', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Uz niske pragove na šumnoj slici točkasti šum ostaje kao lažni dijelovi rubova raspršeni posvuda, posebno u jednolikoj pozadini. Uz najviše pragove slabi detalji, poput prozorskih stakala i finog šrafiranja zavjese i odjeće, potpuno nestaju, dok izražen tamni obris lika i rubovi knjiga ostaju &mdash; stvarni primjer kompromisa Cannyjevih pragova između odbacivanja šuma i osjetljivosti na slabe, ali stvarne rubove.


## Od rubova do pravaca: Houghova transformacija

Detekcija rubova pronalazi *gdje* se intenzitet naglo mijenja, ali ne prepoznaje da raspršeni skup rubnih piksela čini pravac. **Houghova transformacija** (<a href="../references.html#hough-1962">Hough, 1962</a>; <a href="../references.html#duda-hart-1972">Duda & Hart, 1972</a>) odgovara na drukčije pitanje: koje točke iz zadanog skupa rubnih točaka mogu ležati na zajedničkom pravcu?

Pravac ne parametriziramo izrazom $y=mx+b$, koji nije prikladan za okomite pravce, nego udaljenošću od ishodišta i kutom njegove normale:

$$\rho = x\cos\theta + y\sin\theta$$

Za *fiksnu* rubnu točku $(x,y)$ ova jednadžba opisuje sinusoidu u prostoru $(\rho,\theta)$ dok se $\theta$ mijenja &mdash; svaki par $(\rho,\theta)$ na toj krivulji opisuje pravac kroz $(x,y)$. Ključna je ideja da se, ako je više rubnih točaka **kolinearno**, njihove sinusoide sijeku u *istom* paru $(\rho,\theta)$ &mdash; parametrima zajedničkog pravca.


In [ ]:
line_points = [(30, 270), (150, 150), (270, 30)]  # three points on the same line, x + y = 300

# the line's true (rho, theta): its normal direction is (1,1)/sqrt(2), i.e. theta=45 degrees,
# and rho is the line's distance from the origin along that normal
theta_true = np.radians(45)
rho_true = 300 * np.cos(theta_true)

thetas = np.linspace(0, np.pi, 500)
plt.figure(figsize=(6, 3.5))
for x, y in line_points:
    rho = x * np.cos(thetas) + y * np.sin(thetas)
    plt.plot(np.degrees(thetas), rho, label=f'({x},{y})')
plt.axvline(45, color='gray', linestyle='--', linewidth=1)
plt.scatter([np.degrees(theta_true)], [rho_true], color='black', zorder=5, marker='x', s=80,
            label=f'true line: rho={rho_true:.1f}, theta=45°')
plt.xlabel('theta (degrees)')
plt.ylabel('rho')
plt.legend(fontsize=8)
plt.title('Each point traces a sinusoid; collinear points cross at one (rho, theta)')
plt.show()

# confirm all three curves genuinely pass through that point, not just visually
for x, y in line_points:
    rho_at_true_theta = x * np.cos(theta_true) + y * np.sin(theta_true)
    print(f'point ({x:>3},{y:>3}): rho at theta=45 deg = {rho_at_true_theta:.2f}  (true rho = {rho_true:.2f})')

### Akumulator: glasovanje za pravce

U praksi svaki rubni piksel glasuje za cijelu svoju sinusoidu mogućih pravaca $(\rho,\theta)$ u diskretiziranom dvodimenzionalnom polju, **akumulatoru**. Ćelija s najviše glasova predstavlja pravac koji najbolje odgovara podacima. Taj postupak implementiramo od početka za jedan umjetno izrađen pravac i provjeravamo ga funkcijom `cv2.HoughLines`.


In [ ]:
line_img = np.zeros((300, 300), dtype=np.uint8)
cv2.line(line_img, (30, 270), (270, 30), 255, 3)  # the same x+y=300 line as above

ys, xs = np.nonzero(line_img)
theta_bins = np.linspace(0, np.pi, 180, endpoint=False)
max_rho = int(np.hypot(*line_img.shape))
accumulator = np.zeros((2 * max_rho, len(theta_bins)), dtype=np.int32)

for x, y in zip(xs, ys):
    rho_vals = (x * np.cos(theta_bins) + y * np.sin(theta_bins)).astype(int) + max_rho
    accumulator[rho_vals, np.arange(len(theta_bins))] += 1

peak_rho_idx, peak_theta_idx = np.unravel_index(np.argmax(accumulator), accumulator.shape)
manual_rho, manual_theta = peak_rho_idx - max_rho, theta_bins[peak_theta_idx]

cv_lines = cv2.HoughLines(line_img, 1, np.pi / 180, threshold=100)
cv_rho, cv_theta = cv_lines[0, 0]

print(f'true line parameters:    rho={rho_true:.1f}, theta={np.degrees(theta_true):.1f} deg')
print(f'manual accumulator peak: rho={manual_rho}, theta={np.degrees(manual_theta):.1f} deg')
print(f'cv2.HoughLines result:   rho={cv_rho:.0f}, theta={np.degrees(cv_theta):.1f} deg')

# recover two far-apart points on the detected line, to draw it back on the image
a, b = np.cos(cv_theta), np.sin(cv_theta)
x0, y0 = a * cv_rho, b * cv_rho
pt1 = (int(x0 + 1000 * (-b)), int(y0 + 1000 * a))
pt2 = (int(x0 - 1000 * (-b)), int(y0 - 1000 * a))
overlay = cv2.cvtColor(line_img, cv2.COLOR_GRAY2BGR)
cv2.line(overlay, pt1, pt2, (0, 0, 255), 3)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].imshow(line_img, cmap='gray')
axes[0].set_title('Input: single line', fontsize=10)
axes[0].axis('off')

# most bins get only 1-2 votes from individual points' sinusoids, while the peak bin gets one
# vote per point on the line (~240 here) -- on a linear color scale that peak washes out
# everything else, so we log-compress the counts to make the fainter sinusoid trails visible too
axes[1].imshow(np.log1p(accumulator), cmap='hot', aspect='auto',
               extent=[0, 180, -max_rho, max_rho], origin='lower')
axes[1].scatter([np.degrees(theta_true)], [rho_true], edgecolor='cyan', facecolor='none', s=150,
                 linewidth=1.5, label='true (rho, theta)')
axes[1].set_xlabel('theta (degrees)')
axes[1].set_ylabel('rho')
axes[1].legend(fontsize=7)
axes[1].set_title('Accumulator (log-scaled)', fontsize=10)

axes[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
axes[2].set_title('Recovered line (red),\noverlaid on input', fontsize=10)
axes[2].axis('off')
plt.tight_layout()
plt.show()

Kao što vidite, osnovni Houghov algoritam vraća beskonačan pravac (crveni izlaz), a ne dužinu (bijeli ulaz).


### `cv2.HoughLinesP`: praktična inačica

Probabilistička Houghova transformacija vraća i krajnje točke *dužina*, a ne samo beskonačne pravce, te se najčešće koristi u praksi. Testiramo je na prizoru s dvjema linijama poznate orijentacije, na kojem prvo primjenjujemo stvarni Cannyjev detektor rubova. Houghove transformacije rade na kartama rubova, a ne na izvornim slikama.


In [ ]:
shapes_with_lines = np.zeros((300, 300), dtype=np.uint8)
cv2.line(shapes_with_lines, (30, 270), (270, 30), 200, 2)   # true angle: -45 degrees
cv2.line(shapes_with_lines, (50, 50), (250, 50), 200, 2)    # true angle: 0 degrees
cv2.circle(shapes_with_lines, (220, 220), 30, 150, -1)      # a distractor shape, no straight edges

edges = cv2.Canny(shapes_with_lines, 80, 160)
segments = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=60, minLineLength=80, maxLineGap=5)

# color each segment by which of the two true lines (-45 or 0 degrees) it's closer to
# (colors given in BGR, since `vis` gets converted to RGB before display below)
color_diagonal, color_horizontal = (255, 0, 0), (80, 80, 255)  # red, blue

vis = cv2.cvtColor(shapes_with_lines, cv2.COLOR_GRAY2BGR)
print(f'{"segment":>20} {"angle (deg)":>12}')
for x1, y1, x2, y2 in segments.reshape(-1, 4):
    angle = np.degrees(np.arctan2(y2 - y1, x2 - x1))
    print(f'({x1},{y1})-({x2},{y2})'.rjust(20), f'{angle:>12.1f}')
    color = color_diagonal if abs(angle - (-45)) < abs(angle - 0) else color_horizontal
    cv2.line(vis, (x1, y1), (x2, y2), color, 2)

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [shapes_with_lines, edges, vis],
                          ['Original', 'Canny edges', 'Detected segments\n(red = -45°, blue = 0°)']):
    ax.imshow(im, cmap='gray' if im.ndim == 2 else None)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Svaka otkrivena dužina gotovo točno odgovara jednom od dvaju stvarnih kutova linija, −45 i 0 stupnjeva &mdash; svaka linija obično daje dvije ili tri gotovo jednake detekcije umjesto jedne jer nacrtana linija ima konačnu širinu, pa Canny pronalazi rub na *obje* strane poteza, međusobno udaljene nekoliko piksela. Upravo zato praktični postupci temeljeni na Houghovoj transformaciji obično spajaju bliske detekcije u naknadnoj obradi. Kružnica, koja uopće nema ravnih rubova, ispravno ne daje nijednu detekciju pravca.

### Osim pravaca: Houghove kružnice

Ista ideja glasovanja može se proširiti na svaki oblik opisiv malim brojem parametara. Kružnici su potrebna 3 parametra — koordinata središta $x$, koordinata središta $y$ i polumjer $r$ — pa `cv2.HoughCircles` prikuplja glasove u trodimenzionalnom prostoru parametara umjesto u dvodimenzionalnom.


In [ ]:
circle_img = np.zeros((200, 200), dtype=np.uint8)
true_center, true_radius = (100, 100), 50
cv2.circle(circle_img, true_center, true_radius, 255, 1)

detected = cv2.HoughCircles(circle_img, cv2.HOUGH_GRADIENT, dp=1, minDist=50,
                             param1=50, param2=20, minRadius=30, maxRadius=70)
cx, cy, r = detected[0, 0]

print(f'true circle:      center={true_center}, radius={true_radius}')
print(f'detected circle:  center=({cx:.1f}, {cy:.1f}), radius={r:.1f}')

vis = cv2.cvtColor(circle_img, cv2.COLOR_GRAY2BGR)
cv2.circle(vis, (int(cx), int(cy)), int(r), (0, 0, 255), 1)
cv2.drawMarker(vis, (int(cx), int(cy)), (0, 0, 255), cv2.MARKER_CROSS, 10)
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title('Detected circle (red) overlaid on the true one')
plt.axis('off')
plt.show()

### Pogled u unutrašnjost: akumulator za kružnice

Funkcija `cv2.HoughCircles` ne omogućuje pristup svojem internom akumulatoru, ali njegova izrada od početka koristi istu ideju glasovanja kao prethodni akumulator za pravce, uz jednu dodatnu dimenziju: kružnici su potrebni središte *i* polumjer, pa svaki rubni piksel glasuje za svaku trojku $(c_x, c_y, r)$ kojoj može pripadati, za svaki kandidatski polumjer $r$.


In [ ]:
circle_edges = cv2.Canny(circle_img, 50, 100)
edge_ys, edge_xs = np.nonzero(circle_edges)

H, W = circle_img.shape
radii = np.arange(30, 71)
vote_thetas = np.linspace(0, 2 * np.pi, 72, endpoint=False)  # angle around each candidate circle
cos_t, sin_t = np.cos(vote_thetas), np.sin(vote_thetas)

circle_accumulator = np.zeros((len(radii), H, W), dtype=np.int32)
for ri, r in enumerate(radii):
    # for this radius, every edge point votes for every center that's r away from it
    cand_cx = np.round(edge_xs[:, None] - r * cos_t[None, :]).astype(int)
    cand_cy = np.round(edge_ys[:, None] - r * sin_t[None, :]).astype(int)
    valid = (cand_cx >= 0) & (cand_cx < W) & (cand_cy >= 0) & (cand_cy < H)
    np.add.at(circle_accumulator[ri], (cand_cy[valid], cand_cx[valid]), 1)

peak_r_idx, peak_cy, peak_cx = np.unravel_index(np.argmax(circle_accumulator), circle_accumulator.shape)
print(f'true circle:       center={true_center}, radius={true_radius}')
print(f'accumulator peak:  center=({peak_cx}, {peak_cy}), radius={radii[peak_r_idx]}')

radius_vs_cx = circle_accumulator.max(axis=1)  # max over cy
radius_vs_cy = circle_accumulator.max(axis=2)  # max over cx

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(radius_vs_cx, aspect='auto', cmap='hot', extent=[0, W, radii[-1], radii[0]])
axes[0].set_xlabel('cx'); axes[0].set_ylabel('radius')
axes[0].set_title('radius vs. cx (max over cy)', fontsize=10)
axes[1].imshow(radius_vs_cy, aspect='auto', cmap='hot', extent=[0, H, radii[-1], radii[0]])
axes[1].set_xlabel('cy'); axes[1].set_ylabel('radius')
axes[1].set_title('radius vs. cy (max over cx)', fontsize=10)
plt.tight_layout()
plt.show()

Kako bismo izbjegli teškoće prikaza trodimenzionalnog polja, prikazujemo dvodimenzionalne presjeke: najveći broj glasova po $c_y$, što daje prikaz polumjera u odnosu na $c_x$, te po $c_x$, što daje prikaz polumjera u odnosu na $c_y$. Svaki presjek pokazuje svijetlo slovo X: glasovi svake rubne točke opisuju dvije dijagonalne linije kako kandidatski polumjer raste, jednu za središta lijevo od nje i jednu za središta desno. Sve se sijeku u stvarnom središtu i polumjeru, što je upravo najsvjetlija točka svakog prikaza.


### Zadaci

1. Povećajte standardnu devijaciju šuma u `flat_noisy` i ponovno pokrenite usporedbu otpornosti na šum. Mijenja li se relativni poredak četiriju jezgri?
2. Funkcija `cv2.Canny` prihvaća parametar `apertureSize` za svoj interni Sobelov korak, uz zadanu vrijednost 3. Isprobajte `apertureSize=5` i usporedite rezultat sa zadanim na šumnoj slici.
3. Cannyjevu koraku histereze potreban je *povezan* put piksela iznad niskog praga između slabog i jakog ruba. Ručno izradite mali binarni primjer kao NumPy polje u kojem je stvarni rub prekinut na dva dijela prazninom od jednog piksela i provjerite da ih histereza ne uspijeva povezati, iako bi čovjek jasno vidio jedan rub.
4. Smanjite argument `threshold` funkcije `cv2.HoughLinesP` znatno ispod 60 na slici `shapes_with_lines`. Počinju li se pojavljivati lažne kratke dužine zbog šuma na Cannyjevu rubu kružnice, iako ona nema ravnih rubova? Kako povećavanje `minLineLength` pomaže riješiti taj problem?
5. Dodajte treću liniju na `shapes_with_lines` pod malim kutom, primjerice od `(30, 200)` do `(270, 150)`, i predvidite njezin kut prije usporedbe izlaza funkcije `HoughLinesP` sa svojom procjenom.
6. Argument `param2` funkcije `cv2.HoughCircles` približno predstavlja prag broja glasova u akumulatoru, poput argumenta `threshold` funkcije `HoughLines` &mdash; niže vrijednosti daju više kružnica, uključujući više lažnih detekcija. Smanjujte `param2` od 30 do 10 na šumnoj inačici slike `circle_img`, kojoj prvo dodajte Gaussov šum, i pratite kada se počinju pojavljivati lažno pozitivne detekcije kružnica.
